# Week model v2 — last-week lag only

v1 still *had* a yesterday slot (`lag_48`), and filled it with last week on long leads. The tree can keep treating that slot as yesterday.

v2 **drops `demand_lag_48`**. The only demand lag is `demand_lag_336` (last week, known at a Monday freeze). Short lags (`lag_1`, `lag_2`) were never on the week pack. Hour and day packs are unchanged.

Weather and calendar should take more of the |SHAP| share because there is no fake-yesterday knob. Some of that share will move to `lag_336` instead — check the SHAP table, do not assume weather wins automatically.

Writes to `notebooks/week/v2/models`, **not** `data/models`. Set `PROMOTE = True` only after the last-week and holdout tables. Live `run_pack` uses this pack when metadata has `long_lags_only` (no recursive `lag_48` walk).

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "energy_forecast").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
HERE = ROOT / "notebooks" / "week" / "v2"
OUT_DIR = HERE / "models"
sys.path.insert(0, str(ROOT / "src"))

from energy_forecast.evaluate import as_board_scored, board_naive_scale, overall_metrics, score
from energy_forecast.explain import explain_scored, mean_abs_shap, shap_source_share
from energy_forecast.features import FEATURE_COLUMNS, WEEK_LONG_LAG_FEATURES, apply_freeze_time_lags
from energy_forecast.forecast import prepare_history
from energy_forecast.live import WEEK, load_board, load_context_actuals
from energy_forecast.model import (
    LGB_WEEK,
    WEEK_N_ESTIMATORS,
    load_models,
    predict_quantiles,
    split_training_frame,
    train_models,
)
from energy_forecast.modes import MEDIUM, resample_to_mode
from energy_forecast.regions import SERIES_ID
from energy_forecast.review import (
    _attach_frozen_and_weather,
    _ensure_calendar,
    _load_archive_weather,
    _pack_rows,
    _ready_predict_frame,
    _recompute_weather_features,
    week_review_span,
)
from energy_forecast.settlement import LONDON_TZ
from energy_forecast.tune import naive_scale_for_product

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
print("repo", ROOT)
print("write models to", OUT_DIR)
print("dropped", set(FEATURE_COLUMNS) - set(WEEK_LONG_LAG_FEATURES))
print("kept lag", [c for c in WEEK_LONG_LAG_FEATURES if c.startswith("demand_lag")])
prod = load_models()

## 1. History

Same Previous Runs frame as notebook 03. v2 only changes which columns the week trees see.

In [ ]:
history = prepare_history()
train, holdout = split_training_frame(history, features=FEATURE_COLUMNS)
train_v2, holdout_v2 = split_training_frame(history, features=WEEK_LONG_LAG_FEATURES)
scale = naive_scale_for_product(train, "medium")
print("train", len(train), "holdout", len(holdout), "v2 holdout", len(holdout_v2))

## 2. Train week v2

Same `LGB_WEEK` / 450 trees. Slow cell. Does not overwrite `data/models`.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
v2, metrics = train_models(
    history,
    model_dir=OUT_DIR,
    role="week",
    features=WEEK_LONG_LAG_FEATURES,
)
meta_path = OUT_DIR / "metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8"))
meta["long_lags_only"] = True
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("saved", OUT_DIR)
print("features", len(v2.features), "lag_48" in v2.features)
metrics

## 3. Holdout (6-hour grid)

Production on true yesterday is the optimistic number (training regime). Freeze-time is what last week needed. v2 never sees yesterday.

In [ ]:
from energy_forecast.model import load_models as load_from

v2 = load_from(OUT_DIR)


def pack_metrics(frame, models, label):
    scored = resample_to_mode(score(frame, models), MEDIUM)
    out = overall_metrics(scored, scale_mse=scale)
    out.insert(0, "model", label)
    return out

holdout_tbl = pd.concat(
    [
        pack_metrics(holdout, prod, "production / true lag_48"),
        pack_metrics(apply_freeze_time_lags(holdout), prod, "production / freeze-time lag_48"),
        pack_metrics(holdout_v2, v2, "v2 / lag_336 only"),
    ],
    ignore_index=True,
)
holdout_tbl

## 4. Last completed live week

In [ ]:
def shap_ready(frame, models):
    out = _ensure_calendar(frame.copy().reset_index(drop=True))
    out["region_id"] = SERIES_ID
    work = out.drop(columns=["window_start"], errors="ignore")
    rebuilt = _recompute_weather_features(work)
    keys = ["timestamp", "lead_hours"] if "lead_hours" in out.columns else ["timestamp"]
    take = [c for c in models.features if c in rebuilt.columns]
    right = rebuilt.loc[:, list(dict.fromkeys([*keys, *take]))].drop_duplicates(keys, keep="last")
    merged = out.merge(right, on=keys, how="left", suffixes=("", "_rebuilt"))
    for column in models.features:
        rebuilt_col = f"{column}_rebuilt"
        if column not in merged.columns and rebuilt_col in merged.columns:
            merged[column] = merged[rebuilt_col]
        elif rebuilt_col in merged.columns:
            merged[column] = pd.to_numeric(merged[column], errors="coerce").combine_first(
                pd.to_numeric(merged[rebuilt_col], errors="coerce")
            )
    merged["region_id"] = SERIES_ID
    return _ready_predict_frame(merged, models)


start, end = week_review_span(load_board(), load_context_actuals())
board = load_board()
ctx = load_context_actuals()
joined = _attach_frozen_and_weather(
    WEEK,
    _pack_rows(WEEK, board, ctx).dropna(subset=["actual_mw", "p50"]),
    _load_archive_weather(),
)
joined["timestamp"] = pd.to_datetime(joined["timestamp"], utc=True)
start_u, end_u = pd.to_datetime(start, utc=True), pd.to_datetime(end, utc=True)
week = joined.loc[(joined["timestamp"] >= start_u) & (joined["timestamp"] < end_u)].copy().reset_index(drop=True)
week["region_id"] = SERIES_ID
print("completed week", pd.Timestamp(start).tz_convert(LONDON_TZ), "->", pd.Timestamp(end).tz_convert(LONDON_TZ), "n", len(week))

scale_live = board_naive_scale()


def metrics_row(frame, label, climatology):
    scored = as_board_scored(frame, climatology=climatology)
    out = overall_metrics(scored, scale_mse=scale_live)
    out.insert(0, "model", label)
    return out, scored


issued_tbl, _ = metrics_row(week, "issued (board)", prod.climatology)
rows = [issued_tbl]

prod_in = apply_freeze_time_lags(shap_ready(week, prod))
prod_hat = predict_quantiles(prod_in, prod)
prod_week = week.copy()
prod_week[["p10", "p50", "p90"]] = prod_hat[["p10", "p50", "p90"]].to_numpy()
prod_tbl, _ = metrics_row(prod_week, "production + freeze-time lags", prod.climatology)
rows.append(prod_tbl)

v2_in = shap_ready(week, v2)
v2_hat = predict_quantiles(v2_in, v2)
v2_week = week.copy()
v2_week[["p10", "p50", "p90"]] = v2_hat[["p10", "p50", "p90"]].to_numpy()
v2_tbl, _ = metrics_row(v2_week, "v2 / lag_336 only", v2.climatology)
rows.append(v2_tbl)

pd.concat(rows, ignore_index=True)

In [ ]:
local = pd.to_datetime(week["timestamp"], utc=True).dt.tz_convert(LONDON_TZ)
daily = pd.DataFrame(
    {
        "day": local.dt.strftime("%a %d %b"),
        "actual": pd.to_numeric(week["actual_mw"], errors="coerce"),
        "issued": pd.to_numeric(week["p50"], errors="coerce"),
        "prod_freeze": pd.to_numeric(prod_week["p50"], errors="coerce"),
        "v2": pd.to_numeric(v2_week["p50"], errors="coerce"),
    }
)
rows = []
for day, g in daily.groupby("day", sort=False):
    rows.append(
        {
            "day": day,
            "n": int(len(g)),
            "actual": float(g["actual"].mean()),
            "issued": float(g["issued"].mean()),
            "issued MAE": float((g["actual"] - g["issued"]).abs().mean()),
            "prod freeze MAE": float((g["actual"] - g["prod_freeze"]).abs().mean()),
            "v2 MAE": float((g["actual"] - g["v2"]).abs().mean()),
        }
    )
pd.DataFrame(rows)

## 5. SHAP: did weather get the weight?

Holdout sample, production (true lags) vs v2 (no yesterday). Demand-lag share should fall; weather + calendar should rise. If lag share just moves to `lag_336`, last week is still doing yesterday’s job.

In [ ]:
n = 2_500
need_p = [c for c in prod.features if c in holdout.columns]
need_v = [c for c in v2.features if c in holdout_v2.columns]
prod_s = holdout.dropna(subset=["demand_mw", *need_p]).sample(n, random_state=0)
v2_s = holdout_v2.dropna(subset=["demand_mw", *need_v]).sample(n, random_state=0)
if "region_id" not in prod_s.columns:
    prod_s["region_id"] = SERIES_ID
if "region_id" not in v2_s.columns:
    v2_s["region_id"] = SERIES_ID

prod_shap = explain_scored(predict_quantiles(prod_s, prod), prod)["mean"]
v2_shap = explain_scored(predict_quantiles(v2_s, v2), v2)["mean"]
prod_share = shap_source_share(prod_shap).assign(model="production")
v2_share = shap_source_share(v2_shap).assign(model="v2 long lag")
share = pd.concat([prod_share, v2_share], ignore_index=True)
print("Holdout |SHAP| share")
display(share.pivot(index="source", columns="model", values="share"))
print("v2 top features")
display(mean_abs_shap(v2_shap, n=12))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, shap_df, title in (
    (axes[0], prod_shap, "Production (includes lag_48)"),
    (axes[1], v2_shap, "v2 (lag_336 only)"),
):
    rank = mean_abs_shap(shap_df, n=10)
    ax.barh(rank["feature"].iloc[::-1], rank["mean_abs_shap"].iloc[::-1])
    ax.set_title(title)
    ax.set_xlabel("mean |SHAP| (MW)")
fig.tight_layout()

## 6. Promote (optional)

Copies v2 over `data/models`. Hour and day folders are untouched. Live week `run_pack` will see `long_lags_only` and skip recursive `lag_48`.

In [ ]:
PROMOTE = False
if PROMOTE:
    from energy_forecast.paths import DATA_MODELS
    import shutil

    DATA_MODELS.mkdir(parents=True, exist_ok=True)
    for src in OUT_DIR.glob("*"):
        if src.is_file():
            shutil.copy2(src, DATA_MODELS / src.name)
    print("promoted week artefacts to", DATA_MODELS)
else:
    print("skipped promote; artefacts remain in", OUT_DIR)